In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 43.539905,
	"longitude": 4.811397,
	"start_date": "2026-07-02",
	"end_date": "2026-07-16",
	"daily": ["precipitation_hours", "snowfall_sum", "rain_sum", "precipitation_sum", "sunshine_duration", "daylight_duration", "sunset", "sunrise", "weather_code", "temperature_2m_mean", "apparent_temperature_mean", "apparent_temperature_max", "apparent_temperature_min", "wind_speed_10m_max", "wind_gusts_10m_max", "shortwave_radiation_sum", "et0_fao_evapotranspiration", "wind_direction_10m_dominant", "temperature_2m_min", "temperature_2m_max"],
	"timezone": "Europe/Berlin",
}

responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_precipitation_hours = daily.Variables(0).ValuesAsNumpy()
daily_snowfall_sum = daily.Variables(1).ValuesAsNumpy()
daily_rain_sum = daily.Variables(2).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(3).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(4).ValuesAsNumpy()
daily_daylight_duration = daily.Variables(5).ValuesAsNumpy()
daily_sunset = daily.Variables(6).ValuesInt64AsNumpy()
daily_sunrise = daily.Variables(7).ValuesInt64AsNumpy()
daily_weather_code = daily.Variables(8).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(9).ValuesAsNumpy()
daily_apparent_temperature_mean = daily.Variables(10).ValuesAsNumpy()
daily_apparent_temperature_max = daily.Variables(11).ValuesAsNumpy()
daily_apparent_temperature_min = daily.Variables(12).ValuesAsNumpy()
daily_wind_speed_10m_max = daily.Variables(13).ValuesAsNumpy()
daily_wind_gusts_10m_max = daily.Variables(14).ValuesAsNumpy()
daily_shortwave_radiation_sum = daily.Variables(15).ValuesAsNumpy()
daily_et0_fao_evapotranspiration = daily.Variables(16).ValuesAsNumpy()
daily_wind_direction_10m_dominant = daily.Variables(17).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(18).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(19).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["precipitation_hours"] = daily_precipitation_hours
daily_data["snowfall_sum"] = daily_snowfall_sum
daily_data["rain_sum"] = daily_rain_sum
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["sunset"] = daily_sunset
daily_data["sunrise"] = daily_sunrise
daily_data["weather_code"] = daily_weather_code
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["apparent_temperature_mean"] = daily_apparent_temperature_mean
daily_data["apparent_temperature_max"] = daily_apparent_temperature_max
daily_data["apparent_temperature_min"] = daily_apparent_temperature_min
daily_data["wind_speed_10m_max"] = daily_wind_speed_10m_max
daily_data["wind_gusts_10m_max"] = daily_wind_gusts_10m_max
daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
daily_data["et0_fao_evapotranspiration"] = daily_et0_fao_evapotranspiration
daily_data["wind_direction_10m_dominant"] = daily_wind_direction_10m_dominant
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["temperature_2m_max"] = daily_temperature_2m_max

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Coordinates: 43.550086975097656°N 4.872180461883545°E
Elevation: 4.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Hourly data
                          date  temperature_2m
0   2026-07-02 00:00:00+02:00       26.750000
1   2026-07-02 01:00:00+02:00       26.049999
2   2026-07-02 02:00:00+02:00       25.299999
3   2026-07-02 03:00:00+02:00       24.750000
4   2026-07-02 04:00:00+02:00       24.100000
..                        ...             ...
355 2026-07-16 19:00:00+02:00       28.500000
356 2026-07-16 20:00:00+02:00       28.549999
357 2026-07-16 21:00:00+02:00       28.049999
358 2026-07-16 22:00:00+02:00       27.000000
359 2026-07-16 23:00:00+02:00       26.799999

[360 rows x 2 columns]

Daily data
                         date  precipitation_hours  snowfall_sum  rain_sum  \
0  2026-07-02 00:00:00+02:00                  0.0           0.0       0.0   
1  2026-07-03 00:00:00+02:00                  0.0           0.0       0.0   
2  2026-07-04 00:00: